# Predict Student Performance from Game Play

## Is it reasonable to treat session information as sequential data?

The intuition of this notebook is to encode all the rows in a session as sequential data, and then use a Recurrent Neural Networks to predict whether the user for this particular session will answer this question correctly.

If the output is potential, this could tremendously reduce the effort of future engineering, or can become a reliable support for encoding useful features, which can combine with features from statistical analysis to produce a better classifier.

# Read the DataFrame

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [ ]:
# Load the dataset
dtypes = {
    'elapsed_time': np.int32,
    'event_name': 'category', 
    'name': 'category',
    'level': 'category',
    'room_coor_x': np.float32,
    'room_coor_y': np.float32,
    'screen_coor_x': np.float32,
    'screen_coor_y': np.float32,
    'hover_duration': np.float32,
    'text': 'category',
    'fqid': 'category',
    'room_fqid': 'category',
    'text_fqid': 'category',
    'fullscreen': 'category',
    'hq': 'category',
    'music': 'category',
    'level_group': 'category'
}

df = pd.read_csv('/kaggle/input/predict-student-performance-from-game-play/train.csv', dtype=dtypes)

# Print the first 5 rows
df.head()

In [ ]:
df.shape

# Intuition of using LSTM

In specific, taking a `session_id`...

In [ ]:
session_1_df = df[df['session_id'] == 20090312431273200]
session_1_df

...which contains 881 actions recorded. We consider it as a report document of 881 words to process and see whether the prediction made from this document is reliable.

**The strategy for encoding a record into numeric format:**

- Employed columns: `event_name`, `name`, `level`, `room_coor_x`, `room_coor_y`, `screen_coor_x`, `screen_coor_y`, `hover_duration`.
- Set all the null values to 0 since there exists a identification, `event_name`, that shows the reason why these values are zeros.
- Encode all categorical columns (using one-hot encoding).

In [ ]:
df.set_index(['session_id', 'index'], inplace=True)

In [ ]:
df = df[['event_name', 'name', 'level', 'room_coor_x', 'room_coor_y', 'screen_coor_x', 'screen_coor_y', 'hover_duration']]
for col in ['room_coor_x', 'room_coor_y', 'screen_coor_x', 'screen_coor_y', 'hover_duration']:
    # Scaling the coordinates and durations
    df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
    df[col] = df[col].fillna(0)

# One-Hot Encoding and Aggregation

We are using a custom `GetDummies` class for one-hot encoding for 2 reasons:

1. `OneHotEncoder` runs excessively slower than `pd.get_dummies` in encoding large data.
2. `pd.get_dummies` transformations may encounter inconsistent amount of columns in transformed data if 2 datasets contain different number of unique categorical values.

In [ ]:
import sklearn


class GetDummies(sklearn.base.TransformerMixin):
    """Fast one-hot-encoder that makes use of pandas.get_dummies() safely
    on train/test splits.
    """
    def __init__(self, dtypes=None):
        self.input_columns = None
        self.final_columns = None
        if dtypes is None:
            dtypes = [object, 'category']
        self.dtypes = dtypes

    def fit(self, X, y=None, **kwargs):
        self.input_columns = list(X.select_dtypes(self.dtypes).columns)
        X = pd.get_dummies(X, columns=self.input_columns)
        self.final_columns = X.columns
        return self
        
    def transform(self, X, y=None, **kwargs):
        X = pd.get_dummies(X, columns=self.input_columns)
        X_columns = X.columns
        # if columns in X had values not in the data set used during
        # fit add them and set to 0
        missing = set(self.final_columns) - set(X_columns)
        for c in missing:
            X[c] = 0
        # remove any new columns that may have resulted from values in
        # X that were not in the data set when fit
        return X[self.final_columns]
    
    def get_feature_names(self):
        return tuple(self.final_columns)

In [ ]:
get_dummies = GetDummies()
df = get_dummies.fit_transform(df)
df.shape

In [ ]:
# Aggregate data in each session into a 2D numpy array
grouped_data = df.groupby('session_id').apply(lambda x: np.array(x))
grouped_data

# Convert to PyTorch Dataloader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get the numpy array at the given index
        return torch.from_numpy(self.data[idx]).float()

Since the length of each sequence are different, batch transformation is required. For each batch, we will take the session with highest number of actions recorded, let say A, and perform **padding** to other elements by adding zeros to have the shapes of those elements are identical to A.

In [ ]:
def collate_fn_padd(batch):
    """
    Padds batch of variable length

    Note: it converts things ToTensor manually here since the ToTensor transform
    assume it takes in images rather than arbitrary tensors.
    """
    ## Get sequence lengths
    lengths = [t.shape[0] for t in batch]
    try:
        n_features = batch[0].shape[1]
    except:
        n_features = 1
    max_length = max(lengths)
    if max_length == 0:
        max_length += 1
    batch_size = len(lengths)

    padded_tensor = torch.zeros(batch_size, max_length, n_features, dtype=torch.float32)
    for i, val in enumerate(batch):
        l = lengths[i]
        if n_features == 1:
            padded_tensor[i, :l] = val.reshape(-1, 1)
        else:
            padded_tensor[i, :l] = val
    
    return padded_tensor

In [ ]:
# Create an instance of the custom dataset
dataset = MyDataset(grouped_data.values)

# Create a PyTorch DataLoader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn_padd)

# Processing the labels

Now we collect and process the labels...

In [ ]:
# Collect and process the label
label_df = pd.read_csv('/kaggle/input/predict-student-performance-from-game-play/train_labels.csv')

# session_id format: <session>_q<idx> ==> session = <session>
label_df['session'] = label_df.session_id.apply(lambda x: int(x.split('_')[0]) )

# session_id format: <session>_q<idx> ==> question_idx = <idx>
label_df['question_idx'] = label_df.session_id.apply(lambda x: int(x.split('_')[-1][1:]) )
label_df.drop("session_id", axis=1, inplace=True)

# Pivot the table so that the columns are question indices (from 1 to 18),
# indices are session_ids, and value is 0-1 (correct or not)
pivoted_questions = label_df.pivot(columns='question_idx', values='correct', index='session')

# We have a total_score column here just for analysis if needed
pivoted_questions['total_score'] = pivoted_questions.iloc[:, 0:18].sum(axis=1)

# Rename the columns
pivoted_questions.columns = [f'q_{i}' for i in range(1, 19)] + ['total_score']
pivoted_questions

# LSTM model and Training

In [ ]:
# Define the LSTM model
class StackedLSTM(nn.Module):
    def __init__(self, n_layers, n_hidden, n_features, n_embeddings):
        super(StackedLSTM, self).__init__()
        self.embedding = nn.Linear(n_features, n_embeddings)
        self.lstm = nn.LSTM(n_embeddings, n_hidden, n_layers, batch_first=True)
        self.linear = nn.Linear(n_hidden, 18)
        
    def forward(self, x):
        # Pass the input through the Embedding layer
        embed_out = self.embedding(x)

        # Pass the input through the LSTM layers
        lstm_out, _ = self.lstm(embed_out)

        # Get only the last output of the LSTM layer
        out = lstm_out[:, -1, :]
        
        # Flatten the LSTM output and pass it through the linear layer
        out = self.linear(out)
        
        # Apply sigmoid activation function to the output
        out = torch.sigmoid(out)
        
        return out

# Create an instance of the model
n_layers = 3  # Number of LSTM layers
n_hidden = 16  # Number of LSTM units
n_embeddings = 16 # Number of dimension in embedding layer
n_features = 45  # Number of features in each sequence

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = StackedLSTM(n_layers, n_hidden, n_features, n_embeddings).to(device)

In [ ]:
from tqdm import tqdm

# Define number of output labels (number of questions)
n_out = 18

# Define the batch size
batch_size = 32

# Define the number of epochs
n_epochs = 3

# Data size
n_samples = len(grouped_data)

# Define the loss function and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train the model
model.train()
for epoch in range(n_epochs):
    for i, sample in tqdm(enumerate(dataloader)):
        model.zero_grad()
        
        # Get label
        labels = torch.from_numpy(pivoted_questions.iloc[i*batch_size:(i+1)*batch_size, :18].values).float()
        
        sample = sample.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(sample)

        # Compute the loss
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        sample = sample.to('cpu')
        labels = labels.to('cpu')
        
    # Print the loss after every epoch
    print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}')

In [ ]:
# Evaluate the model
pred_list = []
true_list = []

model.eval()
for i, sample in tqdm(enumerate(dataloader)):
    model.zero_grad()
        
    # Get label
    labels = torch.from_numpy(pivoted_questions.iloc[i*batch_size:(i+1)*batch_size, :18].values).float()
    
    sample = sample.to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(sample)

    sample = sample.to('cpu')
    labels = labels.to('cpu')

    pred_list.append(outputs.data.cpu().numpy())
    true_list.append(labels.data.cpu().numpy())

In [ ]:
# Flatten the result dict
test_pred_flattened = np.concatenate(pred_list).ravel()
test_true_flattened = np.concatenate(true_list).ravel()

# Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print(accuracy_score(test_true_flattened, np.round(test_pred_flattened)))
print(precision_score(test_true_flattened, np.round(test_pred_flattened)))
print(recall_score(test_true_flattened, np.round(test_pred_flattened)))

**Generally, black-box RNN can somehow infer the predictions based on action sequence recorded from the user (with 73.2% accuracy on training set). However, there is something I am still wondering is that the loss did not converge (still at a rate of 0.518x), I hope to get any comments for improvement or spotting whether I have made a mistake in this notebook. Thanks for reading!**

In [ ]:
# For test set

# Remove the training set to save RAM
del(df)
del(grouped_data)
del(dataloader)
del(dataset)

# Workflow on test set

In [ ]:
# PROCESSING THE TEST DATASET

# Reading the dataset
test_df = pd.read_csv('/kaggle/input/predict-student-performance-from-game-play/test.csv', dtype=dtypes)

# Preprocess the data with feature selection
test_df.set_index(['session_id', 'index'], inplace=True)
test_df = test_df[['event_name', 'name', 'level', 'room_coor_x', 'room_coor_y', 'screen_coor_x', 'screen_coor_y', 'hover_duration']]
for col in ['room_coor_x', 'room_coor_y', 'screen_coor_x', 'screen_coor_y', 'hover_duration']:
    # Scaling the coordinates and durations
    test_df[col] = (test_df[col] - test_df[col].min()) / (test_df[col].max() - test_df[col].min())
    test_df[col] = test_df[col].fillna(0)

# Perform one-hot encoding
test_df = get_dummies.transform(test_df)
grouped_data = test_df.groupby('session_id').apply(lambda x: np.array(x))

dataset = MyDataset(grouped_data.values)
dataloader = DataLoader(dataset, batch_size=3, shuffle=True, collate_fn=collate_fn_padd)

# Make predictions
pred_list = []

model.eval()
for i, sample in tqdm(enumerate(dataloader)):
    model.zero_grad()
    sample = sample.to(device)
    # Forward pass
    outputs = model(sample)
    sample = sample.to('cpu')
    pred_list.append(outputs.data.cpu().numpy())
    
pred_flattened = np.concatenate(pred_list).ravel()
session_ids = test_df.index.get_level_values('session_id').unique().tolist()

from functools import reduce
session_ids = reduce(lambda x, y: x + [f'{y}_q{i}' for i in range(1, 19)], session_ids, [])

test_result = pd.DataFrame({
    'session_id': session_ids,
    'correct': (pred_flattened > 0.6).astype('int')
})
test_result.head()

# Submission

In [ ]:
import jo_wilder
env = jo_wilder.make_env()
iter_test = env.iter_test()

In [ ]:
for (test, sample_submission) in iter_test:
    
    test_df = test
    test_df.set_index(['session_id', 'index'], inplace=True)

    test_df = test_df[['event_name', 'name', 'level', 'room_coor_x', 'room_coor_y', 'screen_coor_x', 'screen_coor_y', 'hover_duration']]
    for col in ['room_coor_x', 'room_coor_y', 'screen_coor_x', 'screen_coor_y', 'hover_duration']:
        # Scaling the coordinates and durations
        test_df[col] = (test_df[col] - test_df[col].min()) / (test_df[col].max() - test_df[col].min())
        test_df[col] = test_df[col].fillna(0)

    test_df = get_dummies.transform(test_df)
    grouped_data = test_df.groupby('session_id').apply(lambda x: np.array(x))

    dataset = MyDataset(grouped_data.values)
    dataloader = DataLoader(dataset, batch_size=3, shuffle=True, collate_fn=collate_fn_padd)

    # Make predictions
    pred_list = []

    model.eval()
    for i, sample in tqdm(enumerate(dataloader)):
        model.zero_grad()
        sample = sample.to(device)
        # Forward pass
        outputs = model(sample)
        sample = sample.to('cpu')
        pred_list.append(outputs.data.cpu().numpy())

    pred_flattened = np.concatenate(pred_list).ravel()
    session_ids = test_df.index.get_level_values('session_id').unique().tolist()

    from functools import reduce
    session_ids = reduce(lambda x, y: x + [f'{y}_q{i}' for i in range(1, 19)], session_ids, [])

    test_result = pd.DataFrame({
        'session_id': session_ids,
        'correct': (pred_flattened > 0.6).astype('int')
    })
    test_result.head()
    
    env.predict(test_result)

In [ ]:
!head submission.csv